# Custom `wrap_model_call` & `wrap_tool_call`

`wrap_*` hooks are the most powerful middleware: they receive a `request` and a `handler`, and **you decide** when (or whether) to call `handler(request)`. This lets you time, log, cache, rewrite, or short-circuit calls.

```python
@wrap_model_call
def my_hook(request, handler):
    # ... do something before ...
    response = handler(request)   # run the model (or skip it!)
    # ... do something after ...
    return response
```

In [1]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


USER_AGENT environment variable not set, consider setting it to identify your requests.


## 1. Timing & token logging (`wrap_model_call`)
Measure how long each model call takes and log token usage - without touching the agent logic.

In [2]:
import time
from langchain.agents.middleware import wrap_model_call


@wrap_model_call
def time_model_call(request, handler):
    start = time.perf_counter()
    response = handler(request)
    elapsed = time.perf_counter() - start
    msg = response.result[-1]
    print(f"[timing] model call took {elapsed:.2f}s | tokens: {getattr(msg, 'usage_metadata', None)}")
    return response


agent = create_agent(model=llm_noreason, tools=[], middleware=[time_model_call])
agent.invoke(
    {"messages": [HumanMessage(content="Give me one fun fact about octopuses.")]}
)["messages"][-1].pretty_print()

[timing] model call took 1.63s | tokens: {'input_tokens': 22, 'output_tokens': 90, 'total_tokens': 112, 'input_token_details': {}, 'output_token_details': {}}
================================== Ai Message ==================================

Here is a fun fact: **Octopuses have three hearts.**

Two of the hearts (called branchial hearts) pump blood to the gills to pick up oxygen, while the third (the systemic heart) pumps that oxygenated blood to the rest of the body. Interestingly, the systemic heart actually **stops beating** when the octopus swims, which is one reason why they often prefer crawling along the ocean floor to conserve energy!


## 2. Rewriting the model's reply (`wrap_model_call`)
After the model answers, we append a signature. We only rewrite final text replies (not tool-call messages).

In [3]:
@wrap_model_call
def add_signature(request, handler):
    response = handler(request)
    msg = response.result[-1]
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        response.result[-1] = AIMessage(
            content=f"{msg.content}\n\n-- ACME Support Bot",
            id=msg.id,
        )
    return response


agent = create_agent(model=llm_noreason, tools=[], middleware=[add_signature])
agent.invoke(
    {"messages": [HumanMessage(content="How do I reset my password?")]}
)["messages"][-1].pretty_print()

================================== Ai Message ==================================

Since the steps to reset a password depend entirely on **which service or device** you are trying to access, here are the general methods for the most common scenarios:

### 1. For Websites and Apps (e.g., Google, Facebook, Amazon, Email)
Most services follow a similar pattern:
1.  Go to the **login page** of the service.
2.  Look for a link that says **"Forgot password?"**, **"Can't access your account?"**, or **"Reset password"**.
3.  Enter your **username, email address, or phone number** associated with the account.
4.  Check your email inbox or SMS messages for a **verification code** or a **reset link**.
5.  Click the link or enter the code, then follow the prompts to create a new password.

### 2. For Your Computer (Windows)
*   **Microsoft Account**: If you log in with an email, go to the Microsoft account recovery page on another device, verify your identity, and reset it.
*   **Local Account**:


## 3. Intercepting a tool call (`wrap_tool_call`)
Here we **short-circuit** a tool: if a refund is over $100 we never run the tool and return a denial message instead. Small refunds pass through to the real tool.

In [4]:
from langchain_core.messages import ToolMessage
from langchain.agents.middleware import wrap_tool_call


@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order."""
    return f"Refunded ${amount:.2f} for order {order_id}."


@wrap_tool_call
def refund_guard(request, handler):
    if request.tool_call["name"] == "issue_refund":
        # Tool-call args can arrive as strings, so coerce before comparing.
        try:
            amount = float(request.tool_call["args"].get("amount", 0))
        except (TypeError, ValueError):
            amount = 0.0
        if amount > 100:
            return ToolMessage(
                content="Refund denied: amounts over $100 require a manager.",
                tool_call_id=request.tool_call["id"],
            )
    return handler(request)


agent = create_agent(model=llm_noreason, tools=[issue_refund], middleware=[refund_guard])

In [5]:
print("=== Large refund (blocked by the wrapper) ===")
resp = agent.invoke({"messages": [HumanMessage(content="Please refund $250 for order A-1.")]})
for m in resp["messages"]:
    m.pretty_print()

=== Large refund (blocked by the wrapper) ===


================================ Human Message =================================

Please refund $250 for order A-1.
================================== Ai Message ==================================
Tool Calls:
  issue_refund (call_ed5f805b-9297-4998-88c5-23cdfeefac6a)
 Call ID: call_ed5f805b-9297-4998-88c5-23cdfeefac6a
  Args:
    order_id: A-1
    amount: 250
================================= Tool Message =================================

Refund denied: amounts over $100 require a manager.
================================== Ai Message ==================================

The refund for $250 for order A-1 has been denied because amounts over $100 require manager approval.


In [6]:
print("=== Small refund (allowed through) ===")
resp = agent.invoke({"messages": [HumanMessage(content="Please refund $30 for order B-2.")]})
for m in resp["messages"]:
    m.pretty_print()

=== Small refund (allowed through) ===


================================ Human Message =================================

Please refund $30 for order B-2.
================================== Ai Message ==================================
Tool Calls:
  issue_refund (call_1ad95831-2f0f-4757-b25a-54d6d4c115ee)
 Call ID: call_1ad95831-2f0f-4757-b25a-54d6d4c115ee
  Args:
    order_id: B-2
    amount: 30
================================= Tool Message =================================
Name: issue_refund

Refunded $30.00 for order B-2.
================================== Ai Message ==================================

A refund of $30.00 has been successfully issued for order B-2.


## Summary

- `wrap_model_call(request, handler)` wraps the model call - time it, log it, cache it, rewrite `response.result`, or skip the model entirely.
- `wrap_tool_call(request, handler)` wraps each tool call - validate/modify args, monitor, or short-circuit by returning a `ToolMessage` without calling `handler`.
- These two hooks underpin many built-in middleware (retries, fallback, PII, dynamic prompt).